In [32]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

model = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)
model


from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

def read_email_tool(email_id: str) -> str:
    """Simulate reading an email by returning a dummy email content"""
    return f"Email content for {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Simulate sending an email by returning a confirmation message"""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        )
    ]
)

In [33]:
config = {"configurable":{"thread_id": "test-approve"}}
result = agent.invoke({"messages":[HumanMessage(content="Send an email to tejas@gmail.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am.'")]}, config)


In [34]:
result

{'messages': [HumanMessage(content="Send an email to tejas@gmail.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am.'", additional_kwargs={}, response_metadata={}, id='589e35b6-6ea5-4243-97e0-987049640d1c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to tejas@gmail.com with the subject 'Meeting' and the body 'Let's meet tomorrow at 10am.' Let me check the tools provided.\n\nLooking at the functions, there's a send_email_tool. It requires recipient, subject, and body. The parameters are all there: recipient is tejas@gmail.com, subject is Meeting, and the body is as specified. I need to make sure all required fields are included. Yep, all three are required, and the user provided all. So I should call send_email_tool with these parameters. No need for the read_email_tool here since the task is about sending, not reading. Alright, construct the JSON with the function name and arguments.\n", 'tool_calls': [{'id': 'zh

In [35]:
if result.get("__interrupt__"):
    print("Agent is requesting human intervention.")

    result = agent.invoke(
        {
            "decisions": ["approve"]
        },
        config=config
    )

    print(result["messages"][-1].content)

Agent is requesting human intervention.
The email to tejas@gmail.com with the subject "Meeting" has been successfully sent. Let me know if you need to read or send any other emails!


In [36]:
result

{'messages': [HumanMessage(content="Send an email to tejas@gmail.com with subject 'Meeting' and body 'Let's meet tomorrow at 10am.'", additional_kwargs={}, response_metadata={}, id='589e35b6-6ea5-4243-97e0-987049640d1c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to tejas@gmail.com with the subject 'Meeting' and the body 'Let's meet tomorrow at 10am.' Let me check the tools provided.\n\nLooking at the functions, there's a send_email_tool. It requires recipient, subject, and body. The parameters are all there: recipient is tejas@gmail.com, subject is Meeting, and the body is as specified. I need to make sure all required fields are included. Yep, all three are required, and the user provided all. So I should call send_email_tool with these parameters. No need for the read_email_tool here since the task is about sending, not reading. Alright, construct the JSON with the function name and arguments.\n", 'tool_calls': [{'id': 'zh